# Statistical tests for Fig. 5 (Section 4.3)

- Wilcoxon signed-rank test (two-sided) for all pairs, with the p values adjusted by the Holm method in each comparison.
- The rows of all result files are in the same participant order.

In [1]:
import itertools as it

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [2]:
param_dir = "../param"
out_dir = f"{param_dir}/wilcoxon_holm"
user_num = 18
scene_lis = ["office", "supermarket"]
dist_lis = [3, 18, 34]
method_dict = {"I-VT": "ivtl_80deg_160ms_opt", "I-DT": "idt_07deg_160ms_opt", "I-VDT": "ivdt_80deg_07deg_160ms_opt"}

In [3]:
def wilcoxon_holm(data):
    """Run the Wilcoxon signed-rank test for all pairs and adjust the p values with the Holm method."""
    rows = []
    for a, b in it.combinations(data, 2):
        r = stats.wilcoxon(data[a], data[b])
        rows.append({"group1": a, "group2": b, "W": r.statistic, "p_raw": r.pvalue})
    res = pd.DataFrame(rows)
    res["p_adj"] = multipletests(res["p_raw"], method="holm")[1]
    res["sig"] = np.select([res["p_adj"] < 0.01, res["p_adj"] < 0.05], ["**", "*"], "")
    return res

## Absolute error: comparison of the cumulative distances (for each scene and algorithm)

In [4]:
res = []
for scene in scene_lis:
    for name, folder in method_dict.items():
        r = wilcoxon_holm({d: pd.read_csv(f"{param_dir}/exp_distance/{folder}/{d}/result_{scene}.csv")["error"].values for d in dist_lis})
        r.insert(0, "method", name)
        r.insert(0, "scene", scene)
        res.append(r)
fig5_error = pd.concat(res, ignore_index=True)
fig5_error.to_csv(f"{out_dir}/fig5_error_distance.csv", index=False)
fig5_error.round(4)

,scene,method,group1,group2,W,p_raw,p_adj,sig
0,office,I-VT,3,18,14.0,0.0008,0.0025,**
1,office,I-VT,3,34,18.0,0.0019,0.0039,**
2,office,I-VT,18,34,75.0,0.6705,0.6705,
3,office,I-DT,3,18,16.0,0.0013,0.0026,**
4,office,I-DT,3,34,3.0,0.0000,0.0001,**
5,office,I-DT,18,34,48.0,0.1084,0.1084,
6,office,I-VDT,3,18,10.0,0.0003,0.0007,**
7,office,I-VDT,3,34,0.0,0.0000,0.0000,**
8,office,I-VDT,18,34,32.0,0.0182,0.0182,*
9,supermarket,I-VT,3,18,50.0,0.1297,0.3892,


## Number of fixations: comparison of the algorithms (for each scene and cumulative distance)

In [5]:
def count_fixation(scene, folder, d):
    """Return the number of fixations of each participant."""
    path = f"{param_dir}/exp_distance/fixation_result/{scene}_{folder}/{d}"
    return np.array([len(pd.read_csv(f"{path}/{scene}_user{i}.csv", header=None)) for i in range(user_num)])


res = []
for scene in scene_lis:
    for d in dist_lis:
        r = wilcoxon_holm({name: count_fixation(scene, folder, d) for name, folder in method_dict.items()})
        r.insert(0, "distance", d)
        r.insert(0, "scene", scene)
        res.append(r)
fig5_fixation = pd.concat(res, ignore_index=True)
fig5_fixation.to_csv(f"{out_dir}/fig5_fixation_method.csv", index=False)
fig5_fixation.round(4)

,scene,distance,group1,group2,W,p_raw,p_adj,sig
0,office,3,I-VT,I-DT,22.0,0.0040,0.0120,*
1,office,3,I-VT,I-VDT,50.5,0.1415,0.2830,
2,office,3,I-DT,I-VDT,71.5,0.8121,0.8121,
3,office,18,I-VT,I-DT,15.0,0.0010,0.0010,**
4,office,18,I-VT,I-VDT,1.0,0.0003,0.0007,**
5,office,18,I-DT,I-VDT,0.0,0.0000,0.0000,**
6,office,34,I-VT,I-DT,3.0,0.0005,0.0005,**
7,office,34,I-VT,I-VDT,0.0,0.0000,0.0000,**
8,office,34,I-DT,I-VDT,0.0,0.0000,0.0000,**
9,supermarket,3,I-VT,I-DT,1.0,0.0000,0.0000,**
